<a href="https://colab.research.google.com/github/tingtingtingtin/SEARCHME/blob/sazen%2Fml-initial-setup/notebooks/searchme_local_embedding_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Install & Initialize
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!pip install pyspark -q

import time
import pandas as pd
from google.colab import auth
from google.cloud import bigquery
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, pandas_udf, posexplode, expr
from pyspark.sql.types import ArrayType, StringType, FloatType
from sentence_transformers import SentenceTransformer

auth.authenticate_user()
project_id = 'search-me-cs226'
client = bigquery.Client(project=project_id)

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Sazen_Cleaned_Subset_Pipeline") \
    .getOrCreate()

# 2. Define Model Logic
_model_cache = {}

def get_model():
    if "all-minilm" not in _model_cache:
        # Colab Free Tier: CPU is standard. If using T4 GPU, it will still work via sentence-transformers defaults
        _model_cache["all-minilm"] = SentenceTransformer('all-MiniLM-L6-v2')
    return _model_cache["all-minilm"]

@pandas_udf(ArrayType(StringType()))
def chunk_text_udf(texts: pd.Series) -> pd.Series:
    model = get_model()
    max_length, overlap = 256, 30
    stride = max_length - overlap
    all_chunks = []
    for text in texts:
        if not isinstance(text, str) or not text.strip():
            all_chunks.append([])
            continue
        tokens = model.tokenizer(text, add_special_tokens=False, truncation=False)['input_ids']
        chunks = [model.tokenizer.decode(tokens[i : i + max_length])
                  for i in range(0, len(tokens), stride)]
        all_chunks.append(chunks)
    return pd.Series(all_chunks)

@pandas_udf(ArrayType(FloatType()))
def generate_embeddings_udf(texts: pd.Series) -> pd.Series:
    model = get_model()
    # Added batch_size=64 back in for GPU efficiency
    embeddings = model.encode(texts.tolist(), batch_size=64, show_progress_bar=False)
    return pd.Series(embeddings.tolist())

# 3. Pull Data from BigQuery
print("📥 Pulling cleaned_readme_subset from BigQuery...")
query = f"SELECT repo_name, readme_text FROM `{project_id}.searchme_dataset.cleaned_readme_subset`"
raw_pd_df = client.query(query).to_dataframe()
spark_df = spark.createDataFrame(raw_pd_df)
repo_count = spark_df.count()

# 4. Pipeline Execution & Profiling
print(f"🚀 Starting pipeline for {repo_count} repositories...")
start_time = time.time()

# Define transformations (Lazy)
chunked_df = spark_df.withColumn("chunks", chunk_text_udf(col("readme_text"))) \
    .select("repo_name", posexplode(col("chunks")).alias("chunk_idx", "chunk_text")) \
    .withColumn("chunk_id", expr("concat(repo_name, '_chunk_', cast(chunk_idx as string))"))

final_spark_df = chunked_df.withColumn("embedding", generate_embeddings_udf(col("chunk_text")))

# TRIGGER ACTION (The actual work happens here)
print("⚙️ Processing chunks and generating embeddings (Local Mode)...")
execution_start = time.time()
final_pandas_df = final_spark_df.toPandas()
execution_end = time.time()

# 5. Save Results
destination_table = 'searchme_dataset.test_embeddings_cleaned_subset'
print(f"📤 Saving results to {destination_table}...")
final_pandas_df.to_gbq(destination_table, project_id=project_id, if_exists='replace')

# 6. Performance Report
total_time = execution_end - start_time
num_chunks = len(final_pandas_df)
avg_time_per_repo = total_time / repo_count
projected_3m_hours = (avg_time_per_repo * 3_000_000) / 3600

print("\n" + "="*35)
print("📊 PERFORMANCE REPORT")
print("="*35)
print(f"Total Repos Processed: {repo_count}")
print(f"Total Chunks Created:  {num_chunks}")
print(f"Total Pipeline Time:   {total_time:.2f}s")
print(f"Avg Time per Repo:     {avg_time_per_repo:.4f}s")
print("-" * 35)
print(f"💡 PROJECTED 3M RUN:   {projected_3m_hours:.2f} HOURS")
print("="*35)

📥 Pulling cleaned_readme_subset from BigQuery...
🚀 Starting pipeline for 224 repositories...
⚙️ Processing chunks and generating embeddings (Local Mode)...
📤 Saving results to searchme_dataset.test_embeddings_cleaned_subset...


/tmp/ipykernel_1044/4201044701.py:82: FutureWarning: to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq
  final_pandas_df.to_gbq(destination_table, project_id=project_id, if_exists='replace')
100%|██████████| 1/1 [00:00<00:00, 8272.79it/s]


📊 PERFORMANCE REPORT
Total Repos Processed: 224
Total Chunks Created:  1563
Total Pipeline Time:   64.53s
Avg Time per Repo:     0.2881s
-----------------------------------
💡 PROJECTED 3M RUN:   240.07 HOURS


In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# 1. Define a search query
search_query = "interactive map for video games"
print(f"🔍 Searching for: '{search_query}'\n")

# 2. Embed the query using the exact same model
model = get_model() # Using your existing cached model
query_vector = model.encode([search_query])

# 3. Stack the dataframe embeddings into a numpy matrix
doc_vectors = np.stack(final_pandas_df['embedding'].values)

# 4. Calculate Cosine Similarity between the query and all 1563 chunks
similarities = cosine_similarity(query_vector, doc_vectors)[0]

# 5. Get the Top 3 highest scoring chunks
top_3_indices = np.argsort(similarities)[::-1][:3]

for rank, idx in enumerate(top_3_indices, 1):
    row = final_pandas_df.iloc[idx]
    score = similarities[idx]
    print(f"🏆 Rank {rank} | Score: {score:.4f} | Repo: {row['repo_name']}")
    # Print the first 150 characters of the chunk to verify it makes sense
    print(f"📄 Text: {row['chunk_text'][:150]}...\n")

🔍 Searching for: 'interactive map for video games'



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🏆 Rank 1 | Score: 0.4730 | Repo: unused/xenomap
📄 Text: # interactive map for xenoblade chronicles x is an interactive map in first place to find location of enemies while troopmissions, where time is limit...

🏆 Rank 2 | Score: 0.3854 | Repo: unused/xenomap
📄 Text: - [ ] extend the data : troopmission targets, enemies, tyrants, treasures, etc. - [ ] add language support and translations - [ ] add quick contributi...

🏆 Rank 3 | Score: 0.3593 | Repo: gsrka/MapSearchAndroid
📄 Text: # mapsapifullstackproject this is an android application which demonstrates the map search application functionality. the application uses android com...

